In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

err_bound = 1e-4  # 误差界限
h = 1.0  # 网格间距
eps0 = 1.0  # 真空介电常数

In [ ]:
def jacobi_relaxation(phi, rho, h, eps0):
    """
    Jacobi松弛方法
    """
    phi_new = np.copy(phi)
    n = phi.shape[0]
    for i in range(1, n-1):
        for j in range(1, n-1):
            phi_new[i, j] = 0.25 * (phi[i+1, j] + phi[i-1, j] + 
                                   phi[i, j+1] + phi[i, j-1] + 
                                   h**2 * rho[i, j] / eps0) 
    return phi_new

def gauss_seidel_relaxation(phi, rho, h, eps0):
    """
    Gauss-Seidel松弛方法
    """
    phi_new = np.copy(phi)
    n = phi.shape[0]
    
    for i in range(1, n-1):
        for j in range(1, n-1):
            phi_new[i, j] = 0.25 * (phi_new[i+1, j] + phi_new[i-1, j] + 
                                   phi_new[i, j+1] + phi_new[i, j-1] + 
                                   h**2 * rho[i, j] / eps0)
    return phi_new

def sor_relaxation(phi, rho, h, eps0, omega=1.5):
    """
    SOR松弛方法
    """
    phi_new = np.copy(phi)
    n = phi.shape[0]
    
    for i in range(1, n-1):
        for j in range(1, n-1):
            gs_update = 0.25 * (phi_new[i+1, j] + phi_new[i-1, j] + 
                               phi_new[i, j+1] + phi_new[i, j-1] + 
                               h**2 * rho[i, j] / eps0)
            phi_new[i, j] = (1 - omega) * phi_new[i, j] + omega * gs_update
    
    return phi_new

def calc_error(phi_old, phi_new):
    """
    计算两次迭代间的最大误差
    """
    return np.max(np.abs(phi_new - phi_old))

In [ ]:
def solve_poisson(rho, method='jacobi', omega=1.5, max_iter=10000):
    """
    求解泊松方程
    max_iter: 最大迭代次数
    """
    n = rho.shape[0]
    phi = np.zeros((n, n))  # 初始电势为0
    
    iter_count = 0
    error = float('inf')
    
    while error > err_bound and iter_count < max_iter:
        phi_old = np.copy(phi)
        
        if method == 'jacobi':
            phi = jacobi_relaxation(phi, rho, h, eps0)
        elif method == 'gauss_seidel':
            phi = gauss_seidel_relaxation(phi, rho, h, eps0)
        elif method == 'sor':
            phi = sor_relaxation(phi, rho, h, eps0, omega)
        
        error = calc_error(phi_old, phi)
        iter_count += 1
    
    return phi, iter_count

## a. hollow square charge distribution

In [ ]:
def create_hollow_square(N):
    """
    创建空心正方形电荷分布
    N: 外正方形边长参数
    """
    size = 2 * N + 1  # 网格大小
    rho = np.zeros((size, size))
    
    # 计算内外正方形的边界
    outer_min = 0
    outer_max = size - 1
    inner_min = N // 2
    inner_max = size - 1 - N // 2
    
    # 设置电荷分布
    for i in range(size):
        for j in range(size):
            # 在内外正方形之间地方的电荷密度设为1
            if ((i > outer_min and i < outer_max and j > outer_min and j < outer_max) and
                not (i > inner_min and i < inner_max and j > inner_min and j < inner_max)):
                rho[i, j] = 1.0
    
    return rho

print("Part (a): Hollow Square Charge Distribution")
print("=" * 50)

In [ ]:
N_values = [10, 20, 40]

for N in N_values:
    print(f"\nN = {N}")
    print("-" * 20)
    
    rho = create_hollow_square(N)
    
    # 使用三种方法求解，并保存结果
    methods = ['jacobi', 'gauss_seidel', 'sor']
    method_names = ['Jacobi', 'Gauss-Seidel', 'SOR']
    phis = []
    iterations_list = []
    
    for method, name in zip(methods, method_names):
        phi, iterations = solve_poisson(rho, method=method)
        phis.append(phi)
        iterations_list.append(iterations)
        print(f"{name}: {iterations} iterations")
    
    # ===== 修改部分：将四幅图合并为一行四子图 =====
    fig = plt.figure(figsize=(18, 6))          # 适当调整画布宽度
    
    # 1. 电荷分布（子图1）
    ax1 = fig.add_subplot(1, 4, 1, projection='3d')
    n = rho.shape[0]
    x = np.arange(n)
    y = np.arange(n)
    X, Y = np.meshgrid(x, y)
    dx = dy = 0.8
    dz = rho.flatten()
    nonzero = dz > 0
    x_vals = X.flatten()[nonzero]
    y_vals = Y.flatten()[nonzero]
    z_vals = np.zeros_like(x_vals)
    dz_vals = dz[nonzero]
    ax1.bar3d(x_vals, y_vals, z_vals, dx, dy, dz_vals, 
              color='red', alpha=0.7, shade=True)
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Charge')
    ax1.set_title(f'Charge Distribution (N={N})')
    ax1.view_init(elev=30, azim=45)
    
    # 2. Jacobi, Gauss-Seidel, SOR 电势分布（子图2~4）
    for idx, (phi, name) in enumerate(zip(phis, method_names), start=2):
        ax = fig.add_subplot(1, 4, idx, projection='3d')
        X, Y = np.meshgrid(np.arange(n), np.arange(n))
        surf = ax.plot_surface(X, Y, phi, cmap=cm.viridis,
                               linewidth=0, antialiased=True, alpha=0.8)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Potential')
        ax.set_title(f'{name}')
        ax.view_init(elev=30, azim=45)
        # fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
    
    plt.tight_layout()
    plt.show()